In [ ]:
#### PICK ONE: This node for temporary install
SWARMPATH = '/content/'

In [ ]:
#### PICK ONE: *OR* this node for install in google drive
from google.colab import drive
drive.mount('/content/drive')
SWARMPATH = '/content/drive/MyDrive'

In [ ]:
# Install dotnet dependencies
!wget https://dot.net/v1/dotnet-install.sh -O dotnet-install.sh
!chmod +x dotnet-install.sh
!./dotnet-install.sh --channel 8.0

# Install Clouldflared
!wget https://github.com/cloudflare/cloudflared/releases/download/2024.8.2/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# use the chosen path
import os
os.environ['SWARMPATH'] = SWARMPATH
%cd $SWARMPATH

# Colab breaks venv, so, tell swarm to not make a venv
os.environ['SWARM_NO_VENV'] = 'true'

# Download SwarmUI
url = "https://github.com/StMoelter/SwarmUI"
branch = "fix/colab-existing-packages"
!git clone --branch $branch $url

In [ ]:
# Alright, launch it! Watch the output for a Cloudflare URL
%cd $SWARMPATH/SwarmUI/

import runpy
import subprocess
import sys
from pathlib import Path

# Update from the selected repository and branch, including existing Drive installs.
subprocess.run(['git', 'fetch', url, branch], check=True)
subprocess.run(['git', 'reset', '--hard', 'FETCH_HEAD'], check=True)

# Protect this runtime's preinstalled packages before any Python installation.
# Use a fresh GPU runtime; restarting the Python session does not undo package changes.
runpy.run_path('colab/prepare-colab.py', run_name='__main__')

# Restore only missing requirements when reusing an existing backend (including Drive).
requirements = Path('dlbackend/ComfyUI/requirements.txt')
if requirements.is_file():
    subprocess.run([sys.executable, '-s', '-m', 'pip', 'install', '-r', str(requirements)], check=True)

# (Colab stupid-proofing: drive wonks the perms so this needs an aggressive rebuild)
!rm -rf ./src/bin/live_release

!bash ./launch-linux.sh --launch_mode none --cloudflared-path cloudflared